# Lab 2: Text classification, word vectors and verification

**Natural Language Processing and Large Language Models (MDC_53607_EP)**  
Ye Zhu | M2 Data Science IP Paris

This 90-minute lab follows Lecture 2. You will use word counts to score a review, evaluate a supplied logistic classifier, build word-context vectors, and audit a faulty similarity function. There are **five computational Moodle checkpoints, CP1-CP5**, in addition to the two conceptual Quiz 2 questions.

**Prerequisites:** Python functions, loops and basic NumPy operations; the probability and vector concepts from Lecture 2. Preprocessing and training counts are supplied. You do not train a neural network or implement gradient updates in this lab.

Complete the seven small functions marked `TODO`, run the tiny tests, then submit CP1-CP5 in Moodle. The starter notebook intentionally stops at unfinished functions. All other code is supplied.

For Moodle, submit only the requested number. CP3 is an exact integer; use six decimal places for the other answers. Keep full precision internally.

## Setup and data

**Google Colab:** open/upload this notebook in https://colab.research.google.com/, use a CPU runtime, and run the next cell. If prompted, upload **Lab2_Student_Package.zip**. The cell loads the data from that ZIP. A fresh runtime may require another upload.

**Local Jupyter:** unzip the student package and open the notebook beside `data/`. Python 3.9+ and NumPy are required. `requirements.txt` lists the numerical dependency; Jupyter is your notebook application. The notebook makes no dataset/model downloads, uses no API keys and needs no GPU.

The corpus contains **360 training reviews and 72 test reviews**. These are original artificial teaching examples, with repeated templates and deliberate contrasts. They are not a real-world sentiment benchmark. Each row is one short sentence; labels are `0 = negative`, `1 = positive`. A separate file contains 12 diagnostic vector pairs for CP5.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import io
import json
import math
import zipfile
import numpy as np

DATA_DIR = Path('data')
required = ['train.tsv', 'test.tsv', 'settings.json',
            'fixed_logistic_weights.json', 'similarity_audit.json', 'manifest.json']

if not all((DATA_DIR / filename).is_file() for filename in required):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Unzip the lab package and open this notebook beside data/.') from exc
    print('Upload the complete Lab2_Student_Package.zip when prompted.')
    uploaded = files.upload()
    found = False
    for filename, payload in uploaded.items():
        if not zipfile.is_zipfile(io.BytesIO(payload)):
            continue
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            if all('data/' + name in archive.namelist() for name in required):
                DATA_DIR.mkdir(exist_ok=True)
                for name in required:
                    (DATA_DIR / name).write_bytes(archive.read('data/' + name))
                found = True
                break
    if not found:
        raise FileNotFoundError('The uploaded ZIP does not contain the complete Lab 2 data folder.')

manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['version'] == 'lab2-v1', 'Use the supplied lab2-v1 data.'
assert set(manifest['files']) == set(required) - {'manifest.json'}
for filename, info in manifest['files'].items():
    digest = hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest()
    assert digest == info['sha256'], f'{filename} has changed. Restore the supplied file.'

def read_reviews(filename):
    with (DATA_DIR / filename).open(encoding='utf-8', newline='') as stream:
        return [{'id': row['id'], 'label': int(row['label']),
                 'text': row['text'], 'tokens': row['text'].lower().split()}
                for row in csv.DictReader(stream, delimiter='\t')]

train_rows = read_reviews('train.tsv')
test_rows = read_reviews('test.tsv')
settings = json.loads((DATA_DIR / 'settings.json').read_text())
weight_spec = json.loads((DATA_DIR / 'fixed_logistic_weights.json').read_text())
audit_pairs = json.loads((DATA_DIR / 'similarity_audit.json').read_text())['pairs']
print('Data:', manifest['version'], '| NumPy:', np.__version__)
print('Reviews:', len(train_rows), 'training,', len(test_rows), 'test')
print('Example:', train_rows[0]['text'])

### Shared conventions: do not change these

- Lowercase and split on whitespace. Punctuation is already separated: `.` is a token and is retained. Keep stopwords, including `not` and `but`.
- Learn the vocabulary from **training text only**. Keep tokens occurring at least twice; map all other tokens to `<unk>`. Sort the vocabulary alphabetically and use the supplied index mapping.
- Use **raw occurrence counts**, not binary presence or length-normalised features. A repeated word contributes repeatedly.
- Unlike Lab 1, do **not** add `<s>` or `</s>`. Review boundaries limit context windows but are not additional tokens.
- Class order is always `[0, 1]`, meaning `[negative, positive]`. Naive Bayes ties predict class 0. For logistic predictions, class 1 requires probability strictly greater than 0.5; a tie predicts 0.
- Use natural logs for classification scores and cross-entropy; use base-2 logs for PMI. Use float64 arithmetic and do not round intermediate values.
- Training text supplies both class counts and word-context counts. Test labels are used only for evaluation. All parameters in this lab are fixed in advance, so there is no development split or hyperparameter search.

Run the supplied preprocessing cell. `class_token_counts` has shape `(2, V)`, while `class_doc_counts` contains the number of training reviews in each class. These are different denominators.

In [ ]:
UNK = '<unk>'
raw_counts = Counter(w for row in train_rows for w in row['tokens'])
vocab = tuple(sorted({w for w, n in raw_counts.items()
                      if n >= settings['min_training_count']} | {UNK}))
word_to_id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

def encode(tokens):
    return [w if w in word_to_id else UNK for w in tokens]

def bag_of_words(sentences):
    X = np.zeros((len(sentences), V), dtype=np.int64)
    for i, sent in enumerate(sentences):
        for w in sent:
            X[i, word_to_id[w]] += 1
    return X

train = [encode(row['tokens']) for row in train_rows]
test = [encode(row['tokens']) for row in test_rows]
X_train, X_test = bag_of_words(train), bag_of_words(test)
y_train = np.array([r['label'] for r in train_rows], dtype=np.int64)
y_test = np.array([r['label'] for r in test_rows], dtype=np.int64)
class_token_counts = np.vstack([X_train[y_train == c].sum(axis=0) for c in [0, 1]])
class_doc_counts = np.array([(y_train == c).sum() for c in [0, 1]])
test_index = {row['id']: i for i, row in enumerate(test_rows)}

assert X_train.sum() == sum(len(s) for s in train)
assert class_token_counts.sum() == X_train.sum()
assert class_doc_counts.sum() == len(train)
print('Vocabulary size:', V)
print('Class document counts [negative, positive]:', class_doc_counts.tolist())

## 1. Naive Bayes: from counts to a classification score

Let $D_c$ denote the number of training documents belonging to class $c$. Let $P$ denote the probability distributions used by the classifier, where $P(w \mid c)$ is the probability of observing word $w$ in a document of class $c$. Let $C_{cw}$ denote the total number of occurrences of word $w$ across all documents of class $c$. Thus, $\sum_w C_{cw}$ is the total number of word tokens in class $c$, including repetitions.
For class c, use the empirical document prior and add-one word probabilities:

$$P(c)=\frac{D_c}{D_0+D_1},\qquad
P(w\mid c)=\frac{C_{cw}+1}{\sum_v C_{cv}+V}.$$

For a review with count vector x, compute

$$s_c(x)=\log P(c)+\sum_w x_w\log P(w\mid c).$$

Implement `nb_log_scores` and return a length-two array `[s_negative, s_positive]`. Inputs are nonnegative counts, both classes contain training documents, and alpha is positive. This log score omits a class-independent multinomial coefficient, which cancels in the score difference. Do not normalise the scores, drop repeated tokens, or replace the word-token denominator with the document count.

Useful NumPy operations: `sum(axis=1, keepdims=True)`, `np.log`, and matrix-vector multiplication with `@`.

In [ ]:
def nb_log_scores(doc_counts, class_token_counts, class_doc_counts, alpha):
    """Return the two natural-log class scores; rows of class_token_counts are classes 0 and 1."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete nb_log_scores")

In [ ]:
toy_counts = np.array([[2, 1, 0], [0, 1, 3]])
toy_docs = np.array([3, 1])
toy_review = np.array([1, 2, 0])
toy_expected = np.array([
    math.log(3/4) + math.log(3/6) + 2*math.log(2/6),
    math.log(1/4) + math.log(1/7) + 2*math.log(2/7),
])
assert np.allclose(nb_log_scores(toy_review, toy_counts, toy_docs, 1.0), toy_expected, atol=1e-12, rtol=0)
assert np.allclose(nb_log_scores(np.zeros(3), toy_counts, toy_docs, 1.0), np.log([0.75, 0.25]))
print('Naive Bayes toy checks passed.')

### Moodle checkpoint CP1

Using the supplied lab2-v1 training counts, vocabulary and preprocessing, compute the Naive Bayes log-score difference s(positive) - s(negative) for the probe review in settings.json: "the acting was good but the plot was dull dull and lifeless ." Use class document proportions as priors, add-one smoothing (alpha = 1.0) for word probabilities, raw token multiplicities and natural logarithms. Map the probe using the training vocabulary. Enter six digits after the decimal point; retain the sign.

In [ ]:
probe_tokens = encode(settings['nb_probe_text'].split())
probe_counts = bag_of_words([probe_tokens])[0]
probe_scores = nb_log_scores(probe_counts, class_token_counts, class_doc_counts, settings['nb_alpha'])
cp1 = float(probe_scores[1] - probe_scores[0])
print('Encoded probe:', probe_tokens)
print(f'CP1: {cp1:.6f}')

**Discuss:** What does the sign of the score difference tell us? Why are class document counts used for priors, but class token counts for word probabilities?

The following supplied example compares two reviews. Inspect the features and scores before deciding whether this model understands negation.

In [ ]:
contrast_reviews = ['the plot was good not bad .', 'the plot was bad not good .']
contrast_X = bag_of_words([encode(text.split()) for text in contrast_reviews])
assert np.array_equal(contrast_X[0], contrast_X[1])
for text, x in zip(contrast_reviews, contrast_X):
    scores = nb_log_scores(x, class_token_counts, class_doc_counts, settings['nb_alpha'])
    print(text, '-> predicted class', int(np.argmax(scores)))

**Discuss:** Why do these reviews receive identical scores despite their different meanings? Can smoothing resolve this limitation?

## 2. Logistic predictions and cross-entropy

Use the **supplied illustrative weights**, not newly fitted parameters. Unlisted vocabulary weights are zero. The bias must be included:

$$z_i=x_i^\top w+b,\qquad p_i=\sigma(z_i)=\frac{1}{1+e^{-z_i}}.$$

Implement `logistic_forward` to return `(logits, probabilities)`. A stable sigmoid helper is supplied below.

Then implement mean binary cross-entropy for labels y in {0,1} and finite logits z:

$$\overline L=-\frac{1}{n}\sum_i[y_i\ln p_i+(1-y_i)\ln(1-p_i)]
=\frac{1}{n}\sum_i[\log(1+e^{z_i})-y_i z_i].$$

Use `np.logaddexp(0.0, logits)` to calculate `log(1 + exp(logits))` stably. The mean is over reviews in the specified batch. The equivalent logits formula avoids taking the log of rounded probabilities near zero or one. Inputs are nonempty one-dimensional arrays of equal length.

**Interpretation:** Let $y_i$ be the target label of sample $i$. If $y_i = 1$, the loss for that sample is $-\log(p_i)$; if $y_i = 0$, it is $-\log(1-p_i)$. Thus, the loss penalizes confident incorrect predictions: assigning a low probability to class 1 when the true label is 1, or a high probability to class 1 when the true label is 0.

In [ ]:
def stable_sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    ez = np.exp(z[~positive])
    out[~positive] = ez / (1.0 + ez)
    return out

weights = np.array([weight_spec['weights'].get(w, 0.0) for w in vocab], dtype=np.float64)
bias = float(weight_spec['bias'])
batch_indices = [test_index[name] for name in settings['logistic_batch_ids']]
X_batch, y_batch = X_test[batch_indices], y_test[batch_indices]
print('Batch size:', len(y_batch), '| Supplied bias:', bias)

In [ ]:
def logistic_forward(X, weights, bias):
    """Return logits and probabilities for each row of X; use the supplied stable_sigmoid helper."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete logistic_forward")

In [ ]:
def mean_bce_from_logits(labels, logits):
    """Return a Python float: the MEAN binary cross-entropy using natural logarithms."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete mean_bce_from_logits")

In [ ]:
toy_z, toy_p = logistic_forward(np.array([[1, 0], [0, 2]]), np.array([0.5, -0.25]), 0.1)
assert np.allclose(toy_z, [0.6, -0.4])
assert np.allclose(toy_p, [1/(1+math.exp(-0.6)), 1/(1+math.exp(0.4))])
assert math.isclose(mean_bce_from_logits([1, 0], [math.log(3), -math.log(3)]), math.log(4/3), abs_tol=1e-12)
assert math.isclose(mean_bce_from_logits([0, 1], [0, 0]), math.log(2), abs_tol=1e-12)
assert math.isclose(mean_bce_from_logits([1, 0], [1000, -1000]), 0.0, abs_tol=1e-12)
assert math.isclose(mean_bce_from_logits([0, 1], [1000, -1000]), 1000.0, abs_tol=1e-12)
print('Prediction and loss checks passed, including extreme logits.')

### Moodle checkpoint CP2

Using the supplied lab2-v1 fixed_logistic_weights.json and the 12 test IDs in settings.json, what is the MEAN binary cross-entropy? Use raw bag-of-words counts in the fixed training vocabulary, include the supplied bias, take label 1 as positive, and use natural logarithms. Compute from logits with logaddexp(0, z) - y*z. These weights are supplied, not trained in this lab. Enter six digits after the decimal point.

In [ ]:
batch_logits, batch_probabilities = logistic_forward(X_batch, weights, bias)
cp2 = mean_bce_from_logits(y_batch, batch_logits)
print(f'CP2: {cp2:.6f}')
print('Correct predictions in this batch (discussion only):', int(np.sum((batch_probabilities > 0.5) == y_batch)), '/', len(y_batch))

**Discuss:** Can two classifiers have the same accuracy but different cross-entropy? Which is penalised more: a confident wrong prediction or an uncertain wrong prediction? Why have we specified a mean rather than a sum?

These weights were chosen for teaching, not fitted to this corpus. Treat the printed accuracy as an observation about this supplied classifier, not a comparison of properly trained algorithms. Parameter updates are reserved for a later lab.

## 3. Build word-context counts

Ignore the class labels for this part. Build C with shape `(V, V)` from the encoded **training reviews**. A row is a target word and a column is a context word. Use a window of **two positions to the left and two to the right**, with uniform weight 1 for each occurrence.

For every target position i, count context positions j for which `1 <= abs(i-j) <= 2`, within that same review. Exclude the centre position, not all tokens with the same spelling. Repeated words at other positions can therefore create diagonal counts. Count both ordered directions, retain `.` and `<unk>`, and add no boundary tokens.

Example: for `['a', 'b', 'a']` and window 2, there are six ordered events. The matrix has `C[a,a] = 2`, `C[a,b] = 2`, and `C[b,a] = 2`. A separate one-token review contributes no context pairs.

In [ ]:
def count_cooccurrences(sentences, word_to_id, window):
    """Return an integer V-by-V count matrix; respect each review boundary and count token positions."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete count_cooccurrences")

In [ ]:
tiny_ids = {'a': 0, 'b': 1, 'c': 2}
tiny_sentences = [['a', 'b', 'a'], ['c']]
assert np.array_equal(count_cooccurrences(tiny_sentences, tiny_ids, 1), [[0, 2, 0], [2, 0, 0], [0, 0, 0]])
assert np.array_equal(count_cooccurrences(tiny_sentences, tiny_ids, 2), [[2, 2, 0], [2, 0, 0], [0, 0, 0]])
print('Context-window checks passed.')

### Moodle checkpoint CP3

On the encoded lab2-v1 TRAINING reviews, build the word-context matrix with a symmetric window of two token positions on either side. Count every ordered (target, context) occurrence at distances 1 or 2; exclude only the centre position, keep punctuation and repeated tokens, and never cross review boundaries. Do not add sentence-boundary tokens. What is C[good, film], where rows are targets and columns are contexts? Enter an integer.

In [ ]:
C = count_cooccurrences(train, word_to_id, settings['context_window'])
assert C.shape == (V, V) and np.issubdtype(C.dtype, np.integer)
assert np.array_equal(C, C.T), 'A symmetric window should give a symmetric matrix.'
expected_events = sum(2 * sum(max(len(sent) - d, 0) for d in [1, 2]) for sent in train)
assert int(C.sum()) == expected_events
target_word, context_word = settings['count_pair']
cp3 = int(C[word_to_id[target_word], word_to_id[context_word]])
print(f'CP3: {cp3:d}')

**Discuss:** Why is this matrix symmetric? Why can a diagonal entry be nonzero even though we excluded the centre position?

## 4. From raw counts to PPMI and cosine similarity

Frequent contexts can dominate raw count vectors. PMI compares an observed pair with what would be expected from its marginal frequencies. Let

$$N=\sum_{i,j}C_{ij},\qquad r_i=\sum_j C_{ij},\qquad c_j=\sum_i C_{ij}.$$

**Notation:** $C_{ij}$ is the number of times word $j$ appears in the context of word $i$, as defined previously. Thus, $r_i$ is the total number of context occurrences for word $i$ (row sum), while $c_j$ is the total number of times word $j$ appears as a context word (column sum). $N$ is the total number of context occurrences across all word pairs.

For `C[i,j] > 0`, define

$$\operatorname{PPMI}_{ij}=\max\left(0,\log_2\frac{C_{ij}N}{r_i c_j}\right).$$

PMI is the logarithm of the ratio between the observed co-occurrence frequency of two words ($C_{ij}N$) and the expected co-occurrence frequency if they were independent ($r_i c_j$).
PPMI means **positive pointwise mutual information**: negative PMI values are clipped to zero. Set every zero-count entry to zero directly, without evaluating `log(0)`. Zero rows/columns stay zero; an all-zero matrix produces an all-zero result. There is no extra smoothing in this transformation. N counts ordered context events, not words or reviews.

Implement `positive_pmi` and `cosine_similarity`. For cosine, accept equal-length one-dimensional vectors, and raise `ValueError` if either vector has zero norm. The two graded PPMI rows are nonzero. Cosine uses the entire row; do not apply SVD, remove columns or round entries.

$$\cos(u,v)=\frac{u^\top v}{\|u\|_2\|v\|_2}.$$

**Notation and interpretation:** Let $u,v\in\mathbb{R}^V$ denote two rows of the PPMI matrix, for example, $u=\operatorname{PPMI}_{0,\cdot}$ and $v=\operatorname{PPMI}_{1,\cdot}$, where $V$ is the vocabulary size.

A cosine similarity of 1 means that two words have proportional PPMI profiles: they tend to appear in the same contexts, with the same relative associations with other words. A value of 0 means that their profiles share no positive context association.

In [ ]:
def positive_pmi(counts):
    """Return a float64 PPMI matrix of the same shape; use base-2 logs and preserve zero entries."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete positive_pmi")

In [ ]:
def cosine_similarity(u, v):
    """Return cosine similarity as a float; reject zero vectors and incompatible one-dimensional shapes."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete cosine_similarity")

In [ ]:
toy_C = np.array([[2, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]])
toy_P = positive_pmi(toy_C)
toy_expected_P = np.array([[math.log2(3), 0, 0, 0],
                           [0, math.log2(1.5), math.log2(1.5), 0],
                           [0, math.log2(1.5), math.log2(1.5), 0], [0, 0, 0, 0]])
assert np.allclose(toy_P, toy_expected_P, atol=1e-12, rtol=0)
assert np.allclose(positive_pmi(np.array([[1, 3], [3, 1]])),
                   [[0, math.log2(1.5)], [math.log2(1.5), 0]], atol=1e-12, rtol=0)
assert np.array_equal(positive_pmi(np.zeros((2, 2))), np.zeros((2, 2)))
assert math.isclose(cosine_similarity([1, 2], [2, 0]), 1/math.sqrt(5), abs_tol=1e-12)
assert math.isclose(cosine_similarity([1, 0], [0, 1]), 0.0, abs_tol=1e-12)
assert math.isclose(cosine_similarity([1, 0], [-1, 0]), -1.0, abs_tol=1e-12)
try:
    cosine_similarity([0, 0], [1, 0])
    raise AssertionError('A zero vector must raise ValueError.')
except ValueError:
    pass
print('PPMI and cosine checks passed.')

### Moodle checkpoint CP4

Using the lab2-v1 TRAINING co-occurrence matrix from CP3, compute PPMI(i,j) = max(0, log2(C[i,j]*N/(row_sum[i]*column_sum[j]))), with N equal to the sum of all matrix entries and zero-count entries set to zero. What is the cosine similarity between the complete PPMI rows for good and great? Use all vocabulary columns, including punctuation and <unk>, without dimensionality reduction. Enter six digits after the decimal point.

In [ ]:
P = positive_pmi(C)
assert P.shape == C.shape and np.isfinite(P).all() and (P >= 0).all()
first_word, second_word = settings['similarity_pair']
cp4 = cosine_similarity(P[word_to_id[first_word]], P[word_to_id[second_word]])
print(f'CP4: {cp4:.6f}')
print('Raw-count cosine for the same pair (discussion only):', f'{cosine_similarity(C[word_to_id[first_word]], C[word_to_id[second_word]]):.6f}')
print('PPMI cosine: good / bad (discussion only):', f'{cosine_similarity(P[word_to_id["good"]], P[word_to_id["bad"]]):.6f}')

**Discuss:** Why can raw counts give frequent words too much influence? Can `good` and `bad` have similar context vectors despite opposite sentiment? Does a larger cosine value, or applying PPMI, automatically mean a better representation for sentiment classification?

## 5. Audit a plausible similarity function

The candidate below is claimed to compute cosine similarity. On the 12 diagnostic pairs it returns finite scores in `[0, 1]`. That fact alone does not prove correctness.

1. Read the function and run the supplied range check.
2. Design checks using a hand-computable example, swapping the inputs, and multiplying one vector by a positive scalar. Use nonzero vectors of **different lengths in Euclidean norm**, not different numbers of coordinates.
3. Implement `repaired_cosine` independently using scalar products/sums (`math.fsum` is useful). For this exercise, do not call `cosine_similarity` or a library cosine function inside the repair.
4. Compare the original candidate with the repair on the **unmodified** vector pairs in the file. Keep their `u, v` order. Scaling examples are tests, not modifications to the grading data.

Record which check disproved the claim and which line you repaired. This is a short discussion note, not a sixth numerical checkpoint.

In [ ]:
def candidate_similarity(u, v):
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(u)))

candidate_scores = [candidate_similarity(pair['u'], pair['v']) for pair in audit_pairs]
print('All candidate scores finite and in [0, 1]:', all(math.isfinite(x) and 0 <= x <= 1 for x in candidate_scores))
# Add your own checks below. A failed check can be printed as evidence;
# do not leave an intentionally failing assertion that prevents later cells running.

In [ ]:
def repaired_cosine(u, v):
    """Independently compute cosine for equal-length scalar sequences; raise ValueError for zero vectors."""
    # TODO: implement the function following the specification above.
    raise NotImplementedError("Complete repaired_cosine")

In [ ]:
u_check, v_check = [3.0, 4.0], [8.0, 6.0]
assert math.isclose(repaired_cosine(u_check, v_check), 0.96, abs_tol=1e-12)
assert math.isclose(repaired_cosine(u_check, v_check), repaired_cosine(v_check, u_check), abs_tol=1e-12)
assert math.isclose(repaired_cosine(u_check, [3*x for x in v_check]), 0.96, abs_tol=1e-12)
try:
    repaired_cosine([0, 0], [1, 0])
    raise AssertionError('A zero vector must raise ValueError.')
except ValueError:
    pass
print('Independent repair checks passed.')
print('Candidate on the hand-check pair:', candidate_similarity(u_check, v_check))

### Moodle checkpoint CP5

For all 12 ordered pairs in lab2-v1 similarity_audit.json, compare the notebook candidate score dot(u,v)/(norm(u)*norm(u)) with correctly repaired cosine similarity dot(u,v)/(norm(u)*norm(v)). What is the MAXIMUM absolute difference over these 12 pairs? Use the supplied u and v in that order; do not rescale or normalise the inputs before computing the candidate score. Use unrounded intermediate results. Enter six digits after the decimal point.

In [ ]:
corrected_scores = [repaired_cosine(pair['u'], pair['v']) for pair in audit_pairs]
differences = [abs(before - after) for before, after in zip(candidate_scores, corrected_scores)]
cp5 = max(differences)
print(f'CP5: {cp5:.6f}')

**Discuss:** Which checks detect the bug? Why can self-similarity and range checks pass? What evidence would you want before trusting an AI-generated numerical implementation?

## Submit your five computational checkpoints

Copy each number to its matching Moodle question. Follow the instructor's deadline and submit the completed quiz. If the quiz uses deferred feedback, you can revise draft answers before final submission. Keep the notebook as a record of your work.

In [ ]:
submission = {'CP1': cp1, 'CP2': cp2, 'CP3': cp3, 'CP4': cp4, 'CP5': cp5}
for name, value in submission.items():
    print(f'{name}: {value:d}' if name == 'CP3' else f'{name}: {value:.6f}')

## Optional exploration: nearest neighbours

No additional checkpoint. Compare the five nearest neighbours of `good` under raw counts and PPMI. The query itself and zero-norm candidate rows are excluded; exact ties are broken alphabetically. These are tiny-corpus observations, not claims about general English semantics.

In [ ]:
def nearest_neighbors(word, matrix, top_k=5):
    row = matrix[word_to_id[word]]
    if np.linalg.norm(row) == 0:
        raise ValueError('The query has a zero vector.')
    candidates = [(candidate, cosine_similarity(row, matrix[word_to_id[candidate]]))
                  for candidate in vocab if candidate != word and np.linalg.norm(matrix[word_to_id[candidate]]) > 0]
    return sorted(candidates, key=lambda item: (-item[1], item[0]))[:top_k]

for name, matrix in [('Raw counts', C), ('PPMI', P)]:
    print(name)
    for word, similarity in nearest_neighbors('good', matrix):
        print(f'  {word:12s} {similarity:.6f}')

## Closing discussion

- Which operations used class labels, and which learned from text alone?
- Why must the vocabulary and context statistics be built from training text rather than test text?
- Which limitations come from the representation, and which came from incorrect implementation?
- How would you evaluate whether these vectors help a downstream task rather than just producing plausible neighbours?

Course reference: Lecture 2, text classification and word representations. The review corpus and diagnostic fixtures are original teaching material.